## In this notebook, we will create optimization and evaluation splits for all datasets using the proposed methodology

All splits (tuning/eval) are available in `data/splits` folder. This notebook reproduces the methodology used to create them. 

In [1]:
import pandas as pd 
import autorootcwd #noqa\
import numpy as np

In [2]:
from src.data.data_splits import (train_test_split_proposed, ottawa_proposed_split)
from itertools import product
import random

In [3]:
paderborn_metadata = pd.read_pickle(
    "data/metadata/paderborn/files_metadata.bz2"
)
cwru_metadata = pd.read_pickle(
    "data/metadata/cwru/files_metadata.bz2"
)
ottawa_metadata = pd.read_pickle(
    "data/metadata/ottawa/files_metadata.bz2"
)

In [13]:
### PADERBORN ###

class CombinationSampler:
    def __init__(self, list1, list2, list3, random_seed=42):
        """
        Sampler constructor to generate all possible combinations

        Args:
            list1 (list): list of elements
            list2 (list): list of elements
            list3 (list): list of elements
            random_seed (int): random seed
        """
        self.combinations = list(product(list1, list2, list3))
        random.seed(random_seed)
        random.shuffle(self.combinations)

    def get_combination(self, run: int):
        """
        Random combination selection

        Params:
            run (int): run number

        Returns:
            Tuple: random combination
        """
        if not self.combinations:
            raise ValueError("No more combinations available")

        # Select the combination for the current run
        combination = self.combinations[run]

        return combination

healthy_bearing_ids = ["K001", "K002", "K003", "K004", "K005", "K006"]
outer_bearing_ids = ["KA04", "KA15", "KA16", "KA22", "KA30"]
inner_bearing_ids = ["KI04", "KI14", "KI16", "KI18", "KI21", "KI17"]

list1 = [['K001', 'K003'],
    ['K001', 'K002'],
    ['K001', 'K004'],
    ['K001', 'K005'],
    ['K001', 'K006'],
    ['K002', 'K003'],
    ['K002', 'K004'],
    ['K002', 'K005'],
    ['K002', 'K006'],
    ['K003', 'K004'],
    ['K003', 'K005'],
    ['K003', 'K006'],
    ['K004', 'K005'],
    ['K004', 'K006'],
    ['K005', 'K006']]
list2 = [['KA04', 'KA16'],
    ['KA04', 'KA15'],
    ['KA04', 'KA22'],
    ['KA04', 'KA30'],
    ['KA15', 'KA16'],
    ['KA15', 'KA22'],
    ['KA15', 'KA30'],
    ['KA16', 'KA22'],
    ['KA16', 'KA30'],
    ['KA22', 'KA30']]
list3 = [['KI04', 'KI16'],
    ['KI04', 'KI14'],
    ['KI04', 'KI21'],
    ['KI04', 'KI18'],
    ['KI04', 'KI17'],
    ['KI14', 'KI16'],
    ['KI14', 'KI18'],
    ['KI14', 'KI21'],
    ['KI14', 'KI17'],
    ['KI16', 'KI18'],
    ['KI16', 'KI21'],
    ['KI16', 'KI17'],
    ['KI18', 'KI21'],
    ['KI18', 'KI17'],
    ['KI21', 'KI17']]
random_seed = 42

sampler = CombinationSampler(list1, list2, list3, random_seed=random_seed)

for run in range(5):
    splits = train_test_split_proposed(
        paderborn_metadata, run=run, sampler=sampler, healthy_bearing_ids=healthy_bearing_ids,
        outer_bearing_ids=outer_bearing_ids, inner_bearing_ids=inner_bearing_ids, return_ids=True)
    splits = {"train_ids": splits[0], "val_ids": splits[1]}
    pd.DataFrame(pd.Series(splits)).T.to_parquet(f"data/splits/paderborn/tuning/run_{run}.parquet")

for run in range(5, 105):
    splits = train_test_split_proposed(
        paderborn_metadata, run=run, sampler=sampler, healthy_bearing_ids=healthy_bearing_ids,
        outer_bearing_ids=outer_bearing_ids, inner_bearing_ids=inner_bearing_ids, return_ids=True)
    splits = {"train_ids": splits[0], "test_ids": splits[1]}
    pd.DataFrame(pd.Series(splits)).T.to_parquet(f"data/splits/paderborn/eval/run_{run}.parquet")

Test bearings: ['KA15' 'KI17' 'K004' 'KA16' 'KI16' 'K002']
Test bearings: ['KA15' 'KI17' 'K006' 'KI04' 'KA16' 'K002']
Test bearings: ['KA15' 'K003' 'KI04' 'KI18' 'K005' 'KA30']
Test bearings: ['KA15' 'K006' 'K003' 'KI21' 'KI18' 'KA22']
Test bearings: ['KA15' 'K006' 'K003' 'KI04' 'KI14' 'KA04']
Test bearings: ['KI17' 'K003' 'KI14' 'K002' 'KA30' 'KA04']
Test bearings: ['KA15' 'KI04' 'K004' 'KI18' 'K002' 'KA04']
Test bearings: ['K003' 'KI18' 'KI16' 'K002' 'KA30' 'KA22']
Test bearings: ['KI04' 'KI14' 'K005' 'K002' 'KA22' 'KA04']
Test bearings: ['KI17' 'KI21' 'K002' 'KA30' 'K001' 'KA22']
Test bearings: ['K006' 'K004' 'KI18' 'KI16' 'KA22' 'KA04']
Test bearings: ['KA15' 'KI17' 'K003' 'KI18' 'K005' 'KA22']
Test bearings: ['K003' 'KI04' 'KI14' 'KA16' 'K001' 'KA04']
Test bearings: ['KA15' 'K003' 'KI04' 'KI18' 'K001' 'KA04']
Test bearings: ['KA15' 'KI17' 'K006' 'KA16' 'KI16' 'K001']
Test bearings: ['K006' 'KI04' 'KI14' 'K005' 'KA30' 'KA22']
Test bearings: ['KI17' 'K006' 'K004' 'KA16' 'KI16' 'KA04

In [12]:
### OTTAWA ###
for run in range(5):
    splits = ottawa_proposed_split(
        ottawa_metadata,
        n_bearings_per_fault_mode=2,
        run=run,
        random_state=42,
        debug=False,
        return_ids=True,
    )
    splits = {"train_ids": splits[0], "val_ids": splits[1]}
    pd.DataFrame(pd.Series(splits)).T.to_parquet(f"data/splits/uored-vafcls/tuning/run_{run}.parquet")
    
for run in range(5, 105):
    splits = ottawa_proposed_split(
        ottawa_metadata,
        n_bearings_per_fault_mode=2,
        run=run,
        random_state=42,
        debug=False,
        return_ids=True,
    )
    splits = {"train_ids": splits[0], "test_ids": splits[1]}
    pd.DataFrame(pd.Series(splits)).T.to_parquet(f"data/splits/uored-vafcls/eval/run_{run}.parquet")

Selected test IDs: ((1, 5), (8, 9), (13, 14), (16, 18))
Selected test IDs: ((2, 5), (7, 10), (13, 14), (16, 19))
Selected test IDs: ((3, 4), (6, 9), (12, 15), (16, 18))
Selected test IDs: ((1, 2), (8, 9), (12, 15), (16, 17))
Selected test IDs: ((1, 5), (8, 9), (13, 14), (19, 20))
Selected test IDs: ((1, 3), (8, 9), (13, 14), (16, 19))
Selected test IDs: ((3, 4), (7, 9), (11, 12), (19, 20))
Selected test IDs: ((1, 4), (7, 10), (13, 14), (19, 20))
Selected test IDs: ((1, 4), (6, 10), (11, 12), (17, 19))
Selected test IDs: ((4, 5), (6, 9), (11, 13), (17, 19))
Selected test IDs: ((1, 5), (7, 10), (11, 14), (17, 19))
Selected test IDs: ((2, 3), (6, 9), (11, 15), (17, 20))
Selected test IDs: ((4, 5), (6, 7), (11, 12), (17, 19))
Selected test IDs: ((2, 4), (9, 10), (11, 14), (17, 18))
Selected test IDs: ((1, 5), (6, 7), (14, 15), (16, 20))
Selected test IDs: ((3, 4), (7, 9), (14, 15), (16, 17))
Selected test IDs: ((2, 3), (8, 9), (11, 15), (17, 20))
Selected test IDs: ((2, 3), (6, 10), (12, 1

In [6]:
from typing import Tuple, Literal
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
# Optimization splits for CWRU

def k_fold_cwru_proposed(
    df: pd.DataFrame, df_normal: pd.DataFrame = None, remove_hp0: bool = False
) -> Tuple[
    Tuple[pd.DataFrame, pd.DataFrame],
    Tuple[pd.DataFrame, pd.DataFrame],
    Tuple[pd.DataFrame, pd.DataFrame],
]:
    """Prepare CWRU dataset for model selection using k-fold cross-validation with 3 folds.

    - LT Split (Fault location and type): stratified, (Size): random
    """

    df_filtered_ids = pd.read_pickle("data/splits/cwru/filtered_cwru.pkl")
    df = df[df["waveform_id"].isin(df_filtered_ids["waveform_id"])].copy()

    df["fault_type_temp"] = df["fault_type"].str.replace("OR@3", "OR")
    df["fault_type_temp"] = df["fault_type_temp"].str.replace("OR@6", "OR").replace("OR@12", "OR")
    
    df["group"] = (
        df["fault_location"].astype(str)
        + "_"
        + df["fault_type_temp"].astype(str)
        + "_"
        + df["fault_size"].astype(str)
    )

    #df.group = df.fault_size
    # df.drop(columns=["fault_type_temp"], inplace=True)

    if remove_hp0:
        df = df[~(df["load"] == 0)].copy()
        if df_normal is not None:
            df_normal = df_normal[~(df_normal["load"] == 0)].copy()

    df_train_1 = pd.DataFrame()
    df_train_2 = pd.DataFrame()
    df_train_3 = pd.DataFrame()

    df_test_1 = pd.DataFrame()
    df_test_2 = pd.DataFrame()
    df_test_3 = pd.DataFrame()

    gkfold = GroupKFold(n_splits=3, shuffle=True)

    # For each location-type pair, split into train and test by randomly selecting a size
    for group, df_group in df.groupby(["fault_location", "fault_type_temp"]):
        for i, (train_idx, test_idx) in enumerate(
            gkfold.split(df_group, groups=df_group.group)
        ):
            if i == 0:
                df_train_1 = pd.concat(
                    [df_train_1, df_group.iloc[train_idx]], ignore_index=True
                )
                df_test_1 = pd.concat(
                    [df_test_1, df_group.iloc[test_idx]], ignore_index=True
                )
            elif i == 1:
                df_train_2 = pd.concat(
                    [df_train_2, df_group.iloc[train_idx]], ignore_index=True
                )
                df_test_2 = pd.concat(
                    [df_test_2, df_group.iloc[test_idx]], ignore_index=True
                )
            elif i == 2:
                df_train_3 = pd.concat(
                    [df_train_3, df_group.iloc[train_idx]], ignore_index=True
                )
                df_test_3 = pd.concat(
                    [df_test_3, df_group.iloc[test_idx]], ignore_index=True
                )

    if df_normal is not None:
        df_test_1 = pd.concat([df_test_1, df_normal], ignore_index=True)
        df_test_2 = pd.concat([df_test_2, df_normal], ignore_index=True)
        df_test_3 = pd.concat([df_test_3, df_normal], ignore_index=True)

    df_train_1.drop(columns=["fault_type_temp", "group"], inplace=True)
    df_train_2.drop(columns=["fault_type_temp", "group"], inplace=True)
    df_train_3.drop(columns=["fault_type_temp", "group"], inplace=True)
    df_test_1.drop(columns=["fault_type_temp", "group"], inplace=True)
    df_test_2.drop(columns=["fault_type_temp", "group"], inplace=True)
    df_test_3.drop(columns=["fault_type_temp", "group"], inplace=True)

    optimization_folds = {}
    final_folds = [
        (df_train_1, df_test_1),
        (df_train_2, df_test_2),
        (df_train_3, df_test_3),
    ]

    for i, optimization_fold in enumerate(final_folds):
        df_train = optimization_fold[0]
        df_test = optimization_fold[1]

        for side in ["DE", "FE"]:
            if side == "DE":
                df_train = df_train[
                    ~(
                        (df_train["signal_location"] == "FE")
                        & (df_train["fault_location"] == "DE")
                    )
                ]
                df_test = df_test[
                    ~(
                        (df_test["signal_location"] == "DE")
                        & (df_test["fault_location"] == "FE")
                    )
                ]
            elif side == "FE":
                df_train = df_train[
                    ~(
                        (df_train["signal_location"] == "DE")
                        & (df_train["fault_location"] == "FE")
                    )
                ]
                df_test = df_test[
                    ~(
                        (df_test["signal_location"] == "FE")
                        & (df_test["fault_location"] == "DE")
                    )
                ]
            
            optimization_folds[(i+1, side)] = (df_train, df_test)

    return optimization_folds

In [7]:
folds = k_fold_cwru_proposed(
    cwru_metadata)

In [8]:
for key in folds.keys():
    fold_num, side = key
    df_train, df_test = folds[key]
    # Avoid saving new optimization splits since they are not reproducible, due to GroupKFold nature (when fixing a random seed, it keeps a single fault size per fold)
    #df_train.to_parquet(f"data/splits/cwru/tuning/fold_{fold_num}/train_{side}.parquet", index=False)
    #df_test.to_parquet(f"data/splits/cwru/tuning/fold_{fold_num}/test_{side}.parquet", index=False)

In [9]:
pd.read_parquet("data/splits/cwru/tuning/fold_1/test_FE.parquet").head()

,waveform_id,fault_location,load,rpm,fault_type,fault_size,signal_location,fs,duration,bpfo,bpfi,bsf,ftf,inner,outer,ball,multiclass_label
0,B_14_DE_DE_12_3HP,DE,3,1730,B,14,DE,12000,10.178000,3.5848,5.4152,4.7135,0.39828,0,0,1,3
1,B_14_DE_DE_12_0HP,DE,0,1797,B,14,DE,12000,10.153833,3.5848,5.4152,4.7135,0.39828,0,0,1,3
2,B_14_DE_DE_12_2HP,DE,2,1750,B,14,DE,12000,10.165917,3.5848,5.4152,4.7135,0.39828,0,0,1,3
3,B_14_DE_DE_12_1HP,DE,1,1772,B,14,DE,12000,10.178000,3.5848,5.4152,4.7135,0.39828,0,0,1,3
4,IR_7_DE_DE_12_3HP,DE,3,1730,IR,7,DE,12000,10.243083,3.5848,5.4152,4.7135,0.39828,1,0,0,1


In [10]:
from src.data.data_splits import split_cwru_proposed

In [11]:
# Creating test splits for cwru eval

def save_cwru_eval_splits(
    metadata_df: pd.DataFrame,
    run: int,
    train_side: str = "DE",
    pct = 1,
    test_size = 1 / 3
):

    train, _ , test = split_cwru_proposed(
        metadata_df=metadata_df,
        run=run,
        normal_train_side= train_side,
        split_ratio = test_size,
    )
    
    train.to_parquet(f"data/splits/cwru/eval/train_{train_side}_run_{run}.parquet")
    test.to_parquet(f"data/splits/cwru/eval/test_{train_side}_run_{run}.parquet")

for (run, side) in product(
    range(3,53), ["FE", "DE"]):
    save_cwru_eval_splits(
        metadata_df=cwru_metadata,
        run=run,
        train_side=side,
        test_size = 1 / 3
    )

Test set on run 22 matches one of the tuning folds. Recreating split with another seed.
New test set on run 222 does not match any of the tuning folds. Split created successfully.
Test set on run 22 matches one of the tuning folds. Recreating split with another seed.
New test set on run 222 does not match any of the tuning folds. Split created successfully.
